# Урок 31 — Вступ до HTTP: Як Python спілкується з мережею

## HTTP Requests, TCP Lifecycle, Blocking I/O та REST API

**Модуль 4 · Network & Concurrent Systems · Автор: Viktor Nikoriak**

---

# Про що цей урок?

У попередньому уроці ми опустились на самий низький рівень мережевої системи:

- socket
- TCP byte stream
- kernel buffers
- blocking recv()

Тепер ми піднімаємось на рівень вище.

Більшість Python-розробників ніколи не пишуть `socket.recv()` вручну.

Замість цього вони пишуть:

```python
import requests

response = requests.get('https://api.github.com/users/octocat')
data = response.json()
```

І це виглядає простo.

Але за цим одним рядком ховається повний цикл:

- DNS resolution
- TCP 3-Way Handshake
- TLS handshake (для HTTPS)
- HTTP serialization
- OS blocking syscall
- kernel receive buffer
- HTTP response parsing
- JSON deserialization

І головне:

> Поки `requests.get()` виконується — Python process повністю завмирає.

---

# Головна мета уроку

Ти навчишся:

- **Ментально симулювати** повний lifecycle HTTP-запиту
- **Розуміти** різницю між мережевою помилкою та HTTP-помилкою
- **Бачити** де Python зупиняється і чому
- **Писати** production-grade HTTP код із timeout та error handling
- **Дебажити** мережеві проблеми системно, а не навмання

---

# Де це використовується в реальних системах?

| Система | Як використовується HTTP |
| ------- | ------------------------ |
| Мікросервіси | Сервіси спілкуються через REST API |
| Data Pipelines | Завантаження даних з зовнішніх API |
| Web Scraping | Отримання HTML/JSON з веб-сторінок |
| Webhooks | Сервери надсилають events клієнтам |
| Authentication | JWT токени у Authorization header |
| IoT | Сенсори надсилають дані на backend |

---

# Важлива передумова

Цей урок передбачає, що ти вже розумієш:

- що таке socket (Урок 30)
- що таке TCP byte stream
- що таке blocking I/O
- що таке OS kernel buffers

Якщо ні — поверніться до Уроку 30.

# Learning Objectives

Після цього уроку ти зможеш:

## Architecture Understanding

- Пояснити, що насправді відбувається коли Python виконує `requests.get()`
- Описати всі 7 фаз HTTP-запиту від DNS до десеріалізації JSON
- Пояснити різницю між мережевою помилкою (`ConnectionError`) та HTTP-помилкою (`404`)
- Пояснити чому HTTP є stateless протоколом та яку проблему це створює

## Practical Understanding

- Написати production-grade HTTP запит з timeout та error handling
- Правильно обробляти всі типи помилок: `Timeout`, `ConnectionError`, `HTTPError`
- Використовувати `requests.Session()` для збереження стану між запитами
- Правильно передавати параметри: `params=` vs `data=` vs `json=`

## Debugging Understanding

- Інспектувати точний HTTP запит, який Python відправив (`PreparedRequest`)
- Перевіряти `status_code` перед обробкою відповіді
- Читати `response.headers` для діагностики проблем
- Розрізняти коли проблема в мережі, а коли у відповіді сервера

---

# Структура уроку

| Частина | Тема |
| ------- | ---- |
| 1 | Що таке HTTP? Навіщо він існує? |
| 2 | Що реально робить `requests.get()`? |
| 3 | 7 фаз HTTP-запиту: від DNS до JSON |
| 4 | Mental Models: ресторан та листи |
| 5 | Blocking I/O: чому Python завмирає |
| 6 | Production-grade pattern |
| 7 | Edge Cases: 7 пасток для початківців |
| 8 | Debugging: як дебажити HTTP системно |
| 9 | Питання-пастки для перевірки мислення |
| 10 | Практичні вправи |
| 11 | Рефлексія та підсумки |

# 1. Що таке HTTP і навіщо він існує?

HTTP = **HyperText Transfer Protocol**

Це протокол прикладного рівня (Application Layer).

Він визначає:
- **як** клієнт формулює запит
- **як** сервер формулює відповідь
- **який формат** повідомлень
- **які методи** дозволені (GET, POST, PUT, DELETE...)
- **як** передаються метадані (заголовки)

---

## 1.1. Проблема, яку вирішує HTTP

Уяви: у тебе є два сервери на різних континентах.

Вони хочуть обмінятися даними.

TCP вже вирішує транспортування bytes.

Але TCP нічого не знає про:
- що саме запитується
- хто запитує
- у якому форматі дані
- чи успішна операція

HTTP — це **мова**, яку розуміють обидва сервери.

---

## 1.2. HTTP як текстовий протокол

HTTP запит — це просто **форматований ASCII текст**, переданий через TCP socket:

```text
GET /users/octocat HTTP/1.1\r\n
Host: api.github.com\r\n
User-Agent: python-requests/2.28.1\r\n
Accept: */*\r\n
\r\n
```

Це не магія. Це буквально рядок тексту, розбитий на bytes і надісланий через TCP.

---

## 1.3. Де HTTP знаходиться у стеці мережевих протоколів?

```text
┌─────────────────────────────────────────┐
│         Application Layer (L7)          │
│                                         │
│  HTTP · HTTPS · WebSocket · SMTP · DNS  │
│                                         │
│  ← requests, FastAPI, Django, Flask     │
└──────────────────┬──────────────────────┘
                   │ текст/байти
┌──────────────────┴──────────────────────┐
│         Transport Layer (L4)            │
│                                         │
│           TCP · UDP                     │
│                                         │
│  ← socket module, OS kernel             │
└──────────────────┬──────────────────────┘
                   │ сегменти
┌──────────────────┴──────────────────────┐
│         Network Layer (L3)              │
│                                         │
│             IP · ICMP                   │
│                                         │
│  ← IP addresses, routing               │
└──────────────────┬──────────────────────┘
                   │ пакети
┌──────────────────┴──────────────────────┐
│         Physical / Data Link (L1-L2)    │
│                                         │
│          Ethernet · Wi-Fi               │
│                                         │
│  ← електричні / оптичні сигнали        │
└─────────────────────────────────────────┘
```

**Критична ідея:**

Коли ти пишеш `requests.get()`, Python працює лише на рівні L7.

Все інше (TCP, IP, фізика) — це OS Kernel та hardware.

---

## 1.4. HTTP є Stateless

Це одна з найважливіших властивостей HTTP.

> **Stateless = сервер не пам'ятає попередніх запитів**

Кожен HTTP запит:
- повністю незалежний
- повинен містити всю необхідну інформацію
- не може покладатись на "попередній запит"

Аналогія:

> Ти телефонуєш у довідкову службу.
> Відповідає оператор. Ти запитуєш.
> Через 5 хвилин телефонуєш знову.
> Відповідає інший оператор. Він не знає хто ти.
> Тобі треба знову пояснити з початку.

# 2. Що реально робить `requests.get()`?

Більшість початківців думають так:

```python
response = requests.get('https://api.github.com/users/octocat')
```

і уявляють:

```text
Python → Internet → Сервер → Відповідь
```

Але реальність набагато складніша.

---

## 2.1. Шари абстракції у requests

```text
┌────────────────────────────────────────────────────────┐
│  Твій Python код                                       │
│                                                        │
│    requests.get('https://api.github.com/...')          │
└───────────────────────────┬────────────────────────────┘
                            │ high-level wrapper
┌───────────────────────────┴────────────────────────────┐
│  requests library                                      │
│                                                        │
│    Session → PreparedRequest → HTTPAdapter             │
└───────────────────────────┬────────────────────────────┘
                            │ connection pooling
┌───────────────────────────┴────────────────────────────┐
│  urllib3                                               │
│                                                        │
│    ConnectionPool → HTTPConnection → send/recv         │
└───────────────────────────┬────────────────────────────┘
                            │ syscalls
┌───────────────────────────┴────────────────────────────┐
│  OS Kernel (TCP/IP Stack)                              │
│                                                        │
│    DNS · TCP Handshake · TLS · Socket Buffers          │
└───────────────────────────┬────────────────────────────┘
                            │ NIC driver
┌───────────────────────────┴────────────────────────────┐
│  Physical Network                                      │
│                                                        │
│    Ethernet · Wi-Fi · Optical Fiber · Radio waves      │
└────────────────────────────────────────────────────────┘
```

---

## 2.2. Leaky Abstraction

`requests` — це "дірява абстракція" (leaky abstraction).

Вона приховує складність від тебе.

Але якщо щось ламається на нижчому рівні:
- фізична мережа недоступна
- DNS не відповідає
- TCP з'єднання відхилено

Абстракція "протікає" і Python отримує системно-рівневі exceptions:

```text
ConnectionError
Timeout
SSLError
```

---

## 2.3. PreparedRequest — що Python насправді відправляє

Перед відправкою `requests` будує об'єкт `PreparedRequest`.

Це **точне ASCII-представлення** того, що піде по дроту:

```text
GET /users/octocat HTTP/1.1\r\n
Host: api.github.com\r\n
User-Agent: python-requests/2.28.1\r\n
Accept-Encoding: gzip, deflate\r\n
Accept: */*\r\n
Connection: keep-alive\r\n
\r\n
```

Аналогія:

> **Конверт (HTTP Headers)** — метадані про відправника і отримувача.
> Відокремлений від вмісту порожнім рядком `\r\n\r\n`.
>
> **Лист (HTTP Body)** — власне дані (для POST/PUT).
> Для GET запитів body зазвичай порожній.

# 3. 7 Фаз HTTP-запиту: від DNS до JSON

Розберемо кожну фазу виконання:

```python
response = requests.get('https://api.github.com/users/octocat')
```

---

## Фаза 1 — DNS Resolution: Хто такий api.github.com?

```text
[T=0] Python зустрічає requests.get()
```

Перш ніж відправити хоч один байт, комп'ютер повинен знати **IP-адресу** сервера.

```text
Python Process
     │
     │ getaddrinfo("api.github.com")
     ▼
OS Kernel
     │
     │ UDP packet (DNS Query)
     ▼
Local DNS Server (роутер або 8.8.8.8)
     │
     │ DNS Reply: 140.82.112.4
     ▼
OS Kernel → Python Process
```

**Python зупиняється** і чекає відповіді DNS.

---

## Фаза 2 — TCP Handshake: Встановлення з'єднання

```text
[T=1] Python просить OS відкрити TCP socket до 140.82.112.4:443
```

OS Kernel виконує TCP 3-Way Handshake:

```text
Client OS          Server OS
   │                  │
   │──── SYN ────────►│    "Хочу з'єднатись"
   │                  │
   │◄─── SYN-ACK ─────│    "Ок, я готовий"
   │                  │
   │──── ACK ────────►│    "З'єднання встановлено"
   │                  │

Socket State: CLOSED → SYN_SENT → ESTABLISHED
```

Для HTTPS додається ще TLS Handshake (обмін криптографічними ключами).

**Python Process все ще завмер.** Він чекає поки OS повідомить про успіх.

---

## Фаза 3 — Serialization: Будуємо HTTP рядок

```text
[T=2] TCP з'єднання встановлено
```

`requests` будує `PreparedRequest` — точний байтовий рядок:

```text
GET /users/octocat HTTP/1.1\r\n
Host: api.github.com\r\n
User-Agent: python-requests/2.28.1\r\n
Accept: */*\r\n
\r\n
```

---

## Фаза 4 — Transmission & The Long Sleep

```text
[T=3] Python викликає socket.sendall() і відразу socket.recv()
```

```text
Python Memory
    │ байти HTTP запиту
    ▼
OS Kernel Send Buffer
    │ TCP segments
    ▼
NIC (мережева карта)
    │ електричні/оптичні сигнали
    ▼
Internet ...
```

Після `sendall()` Python одразу викликає `recv()` і:

> **PYTHON PROCESS ПОВНІСТЮ ЗАВМИРАЄ**

- CPU використання = 0%
- Scheduler переключається на інші процеси
- Python спить, чекаючи на OS interrupt

---

## Фаза 5 — Server Processing

```text
[T=4] Пакети перетинають Інтернет
```

```text
Client OS Send Buffer
    ═══════════════════════════════════════
    (TCP Packets через Інтернет)
    ═══════════════════════════════════════
Remote OS Receive Buffer
    │
    ▼
Nginx / Web Server
    │
    ▼
Django / FastAPI / Ruby App
    │ SQL запит до БД
    ▼
Database → JSON відповідь
    │
    ▼
Remote OS Send Buffer
```

---

## Фаза 6 — Receiving & Wakeup

```text
[T=5] Електричні сигнали прийшли на нашу мережеву карту
```

```text
NIC отримує пакети
    │ hardware interrupt
    ▼
OS Kernel Receive Buffer (збирає TCP stream)
    │ будить Python process
    ▼
recv() повертається
```

`requests` парсить HTTP відповідь рядок за рядком:

```text
HTTP/1.1 200 OK\r\n
Content-Type: application/json\r\n
Content-Length: 135\r\n
\r\n
{"login": "octocat", "id": 1}
```

Витягує status code (`200`), пакує headers у dict, прикріплює body до `Response` об'єкта.

```text
[T=6] requests.get() нарешті повертає Response об'єкт
```

---

## Фаза 7 — Deserialization: JSON → Python dict

```python
data = response.json()
```

На цьому етапі мережі вже **немає**.

`response.json()` просто бере рядок, який вже знаходиться в RAM:

```text
'{"login": "octocat", "id": 1}'
```

І перетворює його через `json.loads()` на Python словник:

```python
{'login': 'octocat', 'id': 1}
```

---

## Зведена Timeline

```text
[T=0] requests.get() викликано
[T=0] DNS lookup → OS Kernel → DNS Server → IP address
[T=1] TCP 3-Way Handshake (+ TLS для HTTPS)
[T=2] PreparedRequest побудовано (HTTP рядок)
[T=3] socket.sendall() → OS Send Buffer → NIC → Internet
[T=3] socket.recv() → PYTHON PROCESS ЗАВМИРАЄ
[T=4] Сервер обробляє запит (DB queries, business logic)
[T=5] Сервер відповідає → NIC → OS Receive Buffer
[T=5] OS будить Python process
[T=6] HTTP відповідь парситься → Response об'єкт
[T=6] requests.get() повертає Response
[T=7] response.json() → json.loads() → Python dict
```

# 3.1. Mermaid: HTTP Request Lifecycle

```mermaid
sequenceDiagram
    participant PY as Python Process
    participant OS as OS Kernel
    participant DNS as DNS Server
    participant NET as Internet
    participant SRV as Remote Server

    Note over PY: requests.get() викликано

    PY->>OS: getaddrinfo("api.github.com")
    OS->>DNS: UDP DNS Query
    DNS-->>OS: IP: 140.82.112.4
    OS-->>PY: IP address отримано

    PY->>OS: connect(140.82.112.4:443)
    OS->>SRV: SYN
    SRV-->>OS: SYN-ACK
    OS->>SRV: ACK
    Note over OS,SRV: TCP ESTABLISHED

    PY->>OS: socket.sendall(HTTP bytes)
    OS->>NET: TCP Segments
    NET->>SRV: Packets arrive

    Note over PY: PYTHON ЗАВМИРАЄ (blocking recv)

    SRV->>SRV: Business logic, DB query
    SRV->>NET: HTTP Response bytes
    NET->>OS: Packets arrive
    OS->>OS: Receive Buffer заповнено

    Note over PY: OS будить Python process
    OS-->>PY: recv() повертає bytes

    Note over PY: HTTP parsing, Response об'єкт
    Note over PY: response.json() → Python dict
```

---

# 3.2. Mermaid: Шари Абстракції

```mermaid
flowchart TD
    A["Твій код\nrequests.get(url)"] --> B["requests library\nSession · PreparedRequest · HTTPAdapter"]
    B --> C["urllib3\nConnectionPool · HTTPConnection"]
    C --> D["OS Kernel\nTCP/IP Stack · DNS · TLS · Socket Buffers"]
    D --> E["NIC Hardware\nЕлектричні / Оптичні сигнали"]
    E --> F["Internet\nRouters · Switches · Cables"]
    F --> G["Remote Server\nNginx · Django · Database"]

    style A fill:#e8f4f8,stroke:#2196F3
    style B fill:#e8f8e8,stroke:#4CAF50
    style C fill:#fff8e1,stroke:#FF9800
    style D fill:#fce4ec,stroke:#E91E63
    style G fill:#f3e5f5,stroke:#9C27B0
```

In [ ]:
# Встановлення бібліотеки (якщо ще не встановлено)
# !pip install requests

import requests
import json
import time

print(f"requests версія: {requests.__version__}")

# 4. Mental Models: Як мислити про HTTP

## 4.1. Модель Ресторану

Найкраща аналогія для HTTP lifecycle:

```text
┌─────────────────────────────────────────────────────┐
│                 РЕСТОРАН                            │
│                                                     │
│  Клієнт (ти)     →  Замовлення за меню              │
│  Офіціант        →  Мережа (несе замовлення)        │
│  Кухня           →  Сервер (обробляє логіку)        │
│  Страва + рахунок →  HTTP Response (body + headers) │
│  Амнезія кухні   →  HTTP Statelessness              │
└─────────────────────────────────────────────────────┘
```

**Амнезія кухні:**
> Ти замовив каву. Через 5 хвилин хочеш ще одну.
> Кухня не пам'ятає тебе. Треба знову замовити повністю.
> Саме так HTTP statelessness.

---

## 4.2. Модель Поштової Скриньки

```text
Python Process
    │
    │  пишемо листа (серіалізація в HTTP bytes)
    │  кладемо у вихідний лоток (OS Send Buffer)
    │  ставимо будильник (recv() blocking wait)
    │  йдемо спати (Python process заблокований)
    │
    ▼
OS Kernel
    │
    │  поштар забирає лист (NIC driver)
    │  доставляє через пошту (Internet)
    │  чекає відповідь (TCP/IP)
    │
    ▼
Remote Server
    │
    │  отримує листа
    │  обробляє і пише відповідь
    │  відправляє назад
    │
    ▼
OS Kernel
    │
    │  відповідь прийшла (OS Receive Buffer)
    │  будить Python (interrupt)
    │
    ▼
Python Process
    │
    │  прокидається
    │  читає відповідь (Response object)
    │  розпаковує JSON (deserialization)
```

---

## 4.3. Mental Simulation Summary

Коли ти бачиш `requests.get(...)`, уявляй:

> Python кладе листа у вихідний лоток,
> ставить будильник
> і йде спати.
>
> Поки Python спить, OS займається
> складними фізичними деталями:
> фрагментацією пакетів, маршрутизацією, TCP handshakes.
>
> Python прокидається лише коли OS
> стукає в плечo: "конверт із відповіддю прийшов".

# 5. Blocking I/O: Чому Python завмирає

## 5.1. Синхронний vs Асинхронний

`requests` — це **синхронна** бібліотека.

Це означає:

```python
print("До запиту")
response = requests.get('https://httpbin.org/delay/2')  # 2 секунди
print("Після запиту")  # ← цей рядок НЕ виконається 2 секунди
```

Python буквально **не може виконати наступний рядок** поки не прийде відповідь.

---

## 5.2. Чому це так?

```text
[Python виконує requests.get()]
          │
          │ syscall recv()
          ▼
[OS Kernel: перевіряє receive buffer]
          │
          │ buffer порожній
          ▼
[Kernel: process state = SLEEPING]
[Scheduler: забирає CPU у Python]
          │
          │  ...(мілісекунди або секунди)...
          │
[NIC: hardware interrupt!]
[Kernel: копіює bytes у receive buffer]
[Kernel: process state = RUNNABLE]
[Scheduler: повертає CPU Python]
          │
          ▼
[recv() повертає bytes]
[requests.get() повертає Response]
```

---

## 5.3. Інтуїція щодо латентності

```text
CPU: 1 000 000 000 інструкцій/секунду

HTTP запит до сервера в US: ~100 мілісекунд

За ці 100мс CPU міг виконати: 100 000 000 інструкцій

Але Python буквально нічого не робить.
```

Саме через це з'явились:
- `threading` (паралельні потоки)
- `asyncio` (event loop)
- `aiohttp` (async HTTP)

---

## 5.4. The Infinite Hang

> **По замовчуванню `requests.get()` не має timeout!**

Якщо сервер прийняв TCP-з'єднання але ніколи не відповідає:

```python
requests.get('https://httpbin.org/delay/9999')  # буде висіти вічно
```

Python спатиме **нескінченно**.

**Правило production коду:**

```python
# ЗАВЖДИ вказуй timeout!
requests.get(url, timeout=(3.05, 10))
#                          │     │
#                          │     └── read timeout: макс. секунд між пакетами
#                          └──────── connect timeout: макс. секунд на handshake
```

In [ ]:
import requests
import time

# === ДЕМОНСТРАЦІЯ: Blocking I/O ===

print("[1] Перед запитом — Python виконується")

start = time.time()

# httpbin.org/delay/2 — сервер навмисно чекає 2 секунди перед відповіддю
# Python ПОВНІСТЮ ЗУПИНЕНИЙ на цьому рядку
response = requests.get('https://httpbin.org/delay/2', timeout=10)

elapsed = time.time() - start

print(f"[2] Після запиту — Python продовжується")
print(f"[3] Час очікування: {elapsed:.2f} секунд")
print(f"[4] Status code: {response.status_code}")
print()
print("Висновок: Python чекав ~2 секунди і нічого не міг зробити")

In [ ]:
import requests

# === ДЕМОНСТРАЦІЯ: Інспекція PreparedRequest ===
# Подивимось що НАСПРАВДІ Python відправляє по дроту

# Будуємо запит але НЕ відправляємо (prepared request)
session = requests.Session()

request = requests.Request(
    method='GET',
    url='https://httpbin.org/get',
    headers={'X-Custom-Header': 'my-value'},
    params={'search': 'python', 'page': 1}
)

# Серіалізуємо у PreparedRequest (але ще не відправляємо)
prepared = session.prepare_request(request)

print("=== Що Python відправить по дроту ===")
print()
print(f"Метод: {prepared.method}")
print(f"URL (з параметрами): {prepared.url}")
print()
print("=== HTTP Headers (конверт) ===")
for header, value in prepared.headers.items():
    print(f"  {header}: {value}")
print()
print(f"Body: {prepared.body!r}")
print()
print("Зверни увагу:")
print("  params={'search': 'python'} → ?search=python у URL")
print("  Body порожній — GET запити не мають тіла")

In [ ]:
import requests

# === ДЕМОНСТРАЦІЯ: Інспекція того що Python ВІДПРАВИВ ===
# Після реального запиту можна перевірити PreparedRequest

response = requests.get(
    'https://httpbin.org/get',
    params={'q': 'hello'},
    headers={'Authorization': 'Bearer my_token'},
    timeout=5
)

print("=== Що Python реально відправив ===")
print(f"Метод: {response.request.method}")
print(f"URL: {response.request.url}")
print()
print("=== Відправлені Headers ===")
for key, value in response.request.headers.items():
    print(f"  {key}: {value}")
print()
print("=== Відповідь сервера ===")
print(f"Status: {response.status_code}")
print(f"Content-Type: {response.headers.get('Content-Type')}")
print()
print("Правило дебагінгу:")
print("Якщо API відхиляє запит — перевіряй response.request.headers")
print("Саме там видно що Python НАСПРАВДІ відправив")

# 6. Production-Grade Pattern

## 6.1. Мінімальний але глибокий приклад

Ось як виглядає правильний HTTP запит у production коді:

```text
requests.post()
    │
    ├── url        → куди йде запит
    ├── json=      → серіалізація Python dict → JSON body
    ├── headers=   → метадані (авторизація, тип контенту)
    ├── timeout=   → ОБОВ'ЯЗКОВО! захист від infinite hang
    │
    └── try/except блок:
            ├── Timeout       → сервер занадто повільний
            ├── ConnectionError → мережа недоступна / DNS fail
            └── HTTPError     → 4xx / 5xx відповідь
```

In [ ]:
import requests

# === PRODUCTION-GRADE HTTP ЗАПИТ ===

url = "https://httpbin.org/post"

# Заголовки: метадані запиту (аналог "конверту")
headers = {
    "Authorization": "Bearer secret_token_123",
    "Content-Type": "application/json",
}

# Дані: Python dict → автоматично серіалізується в JSON рядок
payload = {
    "sensor_id": 42,
    "temperature": 22.5,
    "location": "Kyiv"
}

try:
    # Крок 1: Серіалізація, мережева передача, blocking wait
    # json=payload → автоматично: dict → JSON рядок → bytes
    # timeout=(3.05, 10) → connect_timeout, read_timeout
    response = requests.post(
        url,
        json=payload,
        headers=headers,
        timeout=(3.05, 10)
    )

    # Крок 2: Перевірка HTTP статусу
    # raise_for_status() кидає HTTPError для 4xx та 5xx
    # БЕЗ ЦЬОГО — програма продовжиться навіть при помилці сервера!
    response.raise_for_status()

    # Крок 3: Десеріалізація (bytes → Python dict)
    # Це вже локальна операція, мережі тут немає
    data = response.json()

    print("Успіх! Сервер отримав дані:")
    print(f"  Наш JSON body: {data.get('json')}")
    print(f"  Наші headers:  {data.get('headers', {}).get('Authorization', 'не знайдено')}")

except requests.exceptions.Timeout:
    # Сервер прийняв з'єднання але не відповів вчасно
    print("Помилка: Мережа занадто повільна — не чекаємо вічно")

except requests.exceptions.ConnectionError:
    # Wi-Fi вимкнено, DNS не відповідає, або порт закрито
    print("Помилка: Мережа недоступна або DNS lookup провалився")

except requests.exceptions.HTTPError as err:
    # Мережа спрацювала (TCP OK), але сервер повернув 4xx або 5xx
    print(f"Помилка: Сервер відхилив запит. Status: {err.response.status_code}")
    print(f"Тіло відповіді: {err.response.text[:200]}")

# 6.2. Що відбулося?

## Схема потоку даних у цьому прикладі

```text
payload = {"sensor_id": 42, ...}
         │
         │ json=payload
         ▼
'{"sensor_id": 42, "temperature": 22.5}'
         │
         │ .encode('utf-8')
         ▼
b'{"sensor_id": 42, "temperature": 22.5}'
         │
         │ HTTP Request body
         ▼
POST /post HTTP/1.1\r\n
Host: httpbin.org\r\n
Content-Type: application/json\r\n
Content-Length: 45\r\n
\r\n
{"sensor_id": 42, "temperature": 22.5}
         │
         │ TCP Socket → Internet → Server
         ▼
Сервер читає body → парсить JSON → обробляє
         │
         │ HTTP Response
         ▼
response.json() → Python dict
```

## Ключові спостереження

1. `json=payload` робить три речі автоматично:
   - серіалізує `dict` → JSON рядок
   - кодує рядок у bytes
   - встановлює `Content-Type: application/json`

2. `raise_for_status()` — **обов'язкова лінія захисту**.
   Без неї програма продовжить виконання навіть якщо сервер повернув 500.

3. `response.json()` — це просто `json.loads(response.text)`.
   Якщо сервер повернув HTML замість JSON — впаде `JSONDecodeError`.

## Типові помилки тут

| Помилка | Наслідок |
| ------- | -------- |
| Забути `timeout=` | Програма може висіти вічно |
| Не викликати `raise_for_status()` | Тихе поглинання помилок сервера |
| `response.json()` без перевірки | `JSONDecodeError` при HTML відповіді |
| Не обробляти `ConnectionError` | Програма падає при відключенні мережі |

# 7. Edge Cases: 7 Пасток для Початківців

---

## Пастка 1: The Infinite Hang

**Симптом:** Jupyter cell висить, CPU = 0%, нічого не відбувається.

```python
# НЕБЕЗПЕЧНО: немає timeout
response = requests.get('https://httpbin.org/delay/60')
```

**Механізм:** TCP handshake успішний, але сервер не відправляє байти.
`recv()` чекає нескінченно — так і задумано OS.

**Виправлення:**
```python
response = requests.get(url, timeout=(3.05, 10))
```

**Реальний impact:** #1 причина каскадних відмов мікросервісів.
Якщо Service A викликає Service B без timeout — при уповільненні B всі
worker threads A зависнуть і A також впаде.

---

## Пастка 2: The Mirage of Success (404/500 — не exceptions!)

**Симптом:** Програма продовжує після отримання 404 або 500.

```python
response = requests.get('https://httpbin.org/status/500')
print("Запит успішно завершено!")  # ← надрукується!
```

**Механізм:** На рівні TCP/IP все спрацювало відмінно. OS доставила bytes.
Той факт що в bytes написано `500 Internal Server Error` —
це деталь application layer. `requests` не осуджує статус-коди за замовчуванням.

**Виправлення:**
```python
response.raise_for_status()  # кидає HTTPError для 4xx/5xx
```

**Реальний impact:** Data pipelines тихо вставляють HTML сторінки помилок у JSON бази даних.

---

## Пастка 3: The JSON Illusion

**Симптом:** `TypeError: string indices must be integers`

```python
data = response.text       # це рядок!
print(data['url'])         # TypeError!
```

**Механізм:** Мережа передає лише bytes. Node.js не знає що таке "Python dict".
Сервер серіалізує у JSON текст. `response.text` — це один великий Python рядок.

**Виправлення:**
```python
data = response.json()     # json.loads(response.text) → dict
print(data['url'])         # OK!
```

---

## Пастка 4: The Amnesiac Server (Stateless HTTP)

**Симптом:** Другий запит отримує 401, хоча перший логін був успішним.

```python
requests.get(url, auth=('user', 'pass'))  # 200 OK
requests.get(url)                          # 401 Unauthorized!
```

**Механізм:** HTTP stateless. Кожен запит ізольований. Сервер не пам'ятає
попередніх запитів. Credentials треба надсилати кожного разу.

**Виправлення:**
```python
session = requests.Session()
session.auth = ('user', 'pass')        # persist across requests
session.get(url)                       # credentials надсилаються автоматично
session.get(other_url)                 # і тут теж
```

---

## Пастка 5: The Silent Redirect

**Симптом:** Один рядок коду виконує **два** HTTP запити.

```python
# http://github.com → 301 → https://github.com
response = requests.get('http://github.com')
print(len(response.history))  # 1 — був редирект!
```

**Механізм:** При отриманні `301 Moved Permanently` requests автоматично
робить новий запит на URL у заголовку `Location`. Дефолтний ліміт: 30 редиректів.

**Перевірка:**
```python
print(response.url)           # кінцевий URL
print(response.history)       # список проміжних відповідей
```

---

## Пастка 6: Bad JSON від Error Pages

**Симптом:** `json.JSONDecodeError` навіть якщо API "має повертати JSON".

```python
response = requests.get('https://httpbin.org/html')
data = response.json()  # JSONDecodeError!
```

**Механізм:** Cloudflare/Nginx перехоплює запит і повертає HTML-сторінку
`502 Bad Gateway` замість JSON бекенду. JSON parser бачить `<` і падає.

**Правильна послідовність:**
```python
response.raise_for_status()                           # спершу перевір статус
if 'application/json' in response.headers.get('Content-Type', ''):
    data = response.json()                           # потім парс JSON
```

---

## Пастка 7: GET + data= = Порожня відповідь

**Симптом:** Сервер нічого не знає про переданий `data=` у GET запиті.

```python
# НЕПРАВИЛЬНО: GET з data у тілі
response = requests.get(url, data={'q': 'python'})
print(response.json()['args'])  # {} — порожньо!

# ПРАВИЛЬНО: GET з params у URL
response = requests.get(url, params={'q': 'python'})
print(response.json()['args'])  # {'q': 'python'}
```

**Механізм:** GET запити за HTTP специфікацією не мають тіла. `params=`
додає параметри у URL (`?q=python`). `data=` йде у тіло — яке сервер ігнорує для GET.

| Аргумент | Куди іде | Коли використовувати |
| -------- | -------- | -------------------- |
| `params=` | URL query string `?k=v` | GET пошук/фільтрація |
| `data=` | HTTP body (form-encoded) | POST форми |
| `json=` | HTTP body (JSON) | POST/PUT API |
| `headers=` | HTTP headers | Авторизація, Content-Type |

In [ ]:
import requests

# === ДЕМОНСТРАЦІЯ ВСІХ EDGE CASES ===

print("=" * 55)
print("ПАСТКА 1: Timeout (захист від infinite hang)")
print("=" * 55)
try:
    # connect_timeout=0.001 — дуже малий, щоб спровокувати timeout
    requests.get('https://httpbin.org/delay/5', timeout=(0.001, 1))
except requests.exceptions.ConnectTimeout:
    print("[OK] ConnectTimeout: захист спрацював — не зависли вічно")
except requests.exceptions.ReadTimeout:
    print("[OK] ReadTimeout: сервер занадто повільний")

print()
print("=" * 55)
print("ПАСТКА 2: 404 — НЕ exception без raise_for_status()")
print("=" * 55)
response = requests.get('https://httpbin.org/status/404', timeout=5)
print(f"Status code: {response.status_code}")
print(f"Python не впав! Перевіряємо вручну...")
try:
    response.raise_for_status()
except requests.exceptions.HTTPError as e:
    print(f"[OK] HTTPError перехоплено: {e}")

print()
print("=" * 55)
print("ПАСТКА 3: response.text vs response.json()")
print("=" * 55)
response = requests.get('https://httpbin.org/get', timeout=5)
text_data = response.text
dict_data = response.json()
print(f"response.text тип: {type(text_data).__name__} (рядок!)")
print(f"response.json() тип: {type(dict_data).__name__} (словник!)")
print(f"dict_data['url']: {dict_data.get('url')}")

print()
print("=" * 55)
print("ПАСТКА 4: ConnectionError — мережева помилка")
print("=" * 55)
try:
    requests.get('http://localhost:9999', timeout=2)
except requests.exceptions.ConnectionError:
    print("[OK] ConnectionError: TCP RST — порт 9999 нічого не слухає")

print()
print("=" * 55)
print("ПАСТКА 5: Silent Redirect")
print("=" * 55)
response = requests.get('https://httpbin.org/redirect/2', timeout=5)
print(f"Кінцевий URL: {response.url}")
print(f"Кількість редиректів: {len(response.history)}")
for i, r in enumerate(response.history):
    print(f"  Редирект {i+1}: {r.status_code} → {r.headers.get('Location')}")

print()
print("=" * 55)
print("ПАСТКА 7: params= vs data= у GET")
print("=" * 55)
# Правильно: params=
response_params = requests.get('https://httpbin.org/get', params={'q': 'python'}, timeout=5)
print(f"params=: URL = {response_params.url}")
print(f"  args: {response_params.json().get('args')}")

# Неправильно: data= у GET
response_data = requests.get('https://httpbin.org/get', data={'q': 'python'}, timeout=5)
print(f"data=:   URL = {response_data.url}")
print(f"  args: {response_data.json().get('args')}  ← порожньо!")

# 8. Debugging: Як Дебажити HTTP Системно

## 8.1. Правильна послідовність дебагінгу

**Типова помилка початківця:**
```python
data = response.json()  # відразу парсить — може впасти!
```

**Правильна послідовність:**

```text
Крок 1: Перевір status_code
    response.status_code

Крок 2: Перевір Content-Type заголовок
    response.headers['Content-Type']

Крок 3: Перегляни перші 200 символів body
    response.text[:200]

Крок 4: Перевір що ти НАСПРАВДІ відправив
    response.request.headers
    response.request.body

Крок 5: Тільки тепер парси JSON
    response.json()
```

---

## 8.2. Типові сценарії та діагностика

| Симптом | Причина | Діагностика |
| ------- | ------- | ----------- |
| `ConnectionError` | Wi-Fi/DNS | `ping domain.com` у терміналі |
| `Timeout` | Повільний сервер | Зменши timeout, перевір доступність |
| `HTTPError 401` | Неправильна авторизація | Перевір `request.headers['Authorization']` |
| `HTTPError 403` | User-Agent заблоковано | Перевір `request.headers['User-Agent']` |
| `JSONDecodeError` | Сервер повернув HTML | Перевір `response.headers['Content-Type']` |
| Порожній `args` у GET | Використав `data=` замість `params=` | Перевір `request.url` |
| `401` на другому запиті | HTTP stateless | Використай `requests.Session()` |

---

## 8.3. Inspect What You Sent, Not What You Wrote

> Якщо API відхиляє твій запит, баг часто у headers або форматуванні.
> Перевіряй `response.request.headers` — це точний рядок, який Python відправив по дроту.

Обходь Python і перевіряй на системному рівні:

```bash
# Перевірка DNS (чи резолвиться домен?)
ping api.github.com

# Детальна HTTP діагностика (що йде по дроту)
curl -v https://api.github.com/users/octocat

# Перевірка доступності порту
telnet api.github.com 443
```

In [ ]:
import requests

# === DEBUGGING TOOLKIT ===

def debug_http_request(url: str, **kwargs) -> requests.Response | None:
    """Повна діагностика HTTP запиту."""

    kwargs.setdefault('timeout', 5)

    try:
        response = requests.get(url, **kwargs)

        print("=== Що Python відправив ===")
        print(f"  Метод:    {response.request.method}")
        print(f"  URL:      {response.request.url}")
        print(f"  Body:     {response.request.body!r}")
        print("  Headers відправлені:")
        for k, v in response.request.headers.items():
            print(f"    {k}: {v}")

        print()
        print("=== Що сервер відповів ===")
        print(f"  Status:       {response.status_code}")
        print(f"  Content-Type: {response.headers.get('Content-Type', '?')}")
        print(f"  Body (перші 300 символів):")
        print(f"    {response.text[:300]}")

        # Редиректи
        if response.history:
            print()
            print(f"  Було {len(response.history)} редирект(ів):")
            for r in response.history:
                print(f"    {r.status_code} → {r.headers.get('Location')}")

        return response

    except requests.exceptions.Timeout:
        print("[TIMEOUT] Сервер не відповів вчасно")
    except requests.exceptions.ConnectionError as e:
        print(f"[CONNECTION ERROR] Мережева помилка: {e}")

    return None


# Запускаємо діагностику
r = debug_http_request(
    'https://httpbin.org/get',
    params={'query': 'python', 'page': 1},
    headers={'X-Custom': 'debug-mode'}
)

# 9. Питання-Пастки для Перевірки Мислення

Спробуй відповісти ДО того як запустити код.

Ці питання спеціально побудовані навколо хибних уявлень.

---

## Питання 1: The "Silent" Failure

```python
import requests
response = requests.get("https://httpbin.org/status/404")
print("Request finished!")
```

**Що станеться?**

- A) Програма впаде з `HTTPError`
- B) Програма надрукує "Request finished!"
- C) Програма зависне вічно

---

## Питання 2: The JSON Illusion

```python
response = requests.get("https://httpbin.org/get")
data = response.text
print(data['url'])
```

**Що станеться?**

- A) Надрукує URL
- B) `TypeError: string indices must be integers`
- C) Надрукує порожній рядок

---

## Питання 3: The Amnesiac Server

```python
requests.get("https://httpbin.org/basic-auth/user/pass", auth=('user', 'pass'))
response = requests.get("https://httpbin.org/basic-auth/user/pass")
print(response.status_code)
```

**Що надрукується?**

- A) `200` — ми вже авторизувались
- B) `401` — HTTP stateless
- C) `ConnectionError`

---

## Питання 4: The Great Freeze

```python
import time
start = time.time()
for i in range(3):
    requests.get('https://httpbin.org/delay/1', timeout=10)
print(f"Час: {time.time() - start:.1f}с")
```

**Скільки часу займе?**

- A) ~1 секунда (паралельно)
- B) ~3 секунди (послідовно)
- C) Миттєво

---

## Питання 5: GET + data=

```python
response = requests.get("https://httpbin.org/get", data={'q': 'python'})
print(response.json()['args'])
```

**Що надрукується?**

- A) `{'q': 'python'}`
- B) `{}` (порожньо)
- C) `HTTPError 405`

---

## Питання 6: Port Mystery

**Чому ми пишемо `:8000` у localhost, але ніколи не пишемо порт для google.com?**

- A) Google використовує секретні порти
- B) Локальний розробник вибрав нестандартний порт; стандартний HTTP=80, HTTPS=443 приховані автоматично
- C) Браузери блокують інші порти

In [ ]:
import requests

# === ВІДПОВІДІ на Питання-Пастки ===

print("=" * 55)
print("ПИТАННЯ 1: Відповідь B — програма продовжує!")
print("=" * 55)
response = requests.get("https://httpbin.org/status/404", timeout=5)
print("Request finished!")
print(f"Status: {response.status_code}")
print("Чому? TCP успішний. 404 — це HTTP деталь, не мережева помилка.")
print("Виправлення: response.raise_for_status()")

print()
print("=" * 55)
print("ПИТАННЯ 2: Відповідь B — TypeError")
print("=" * 55)
response = requests.get("https://httpbin.org/get", timeout=5)
print(f"response.text тип: {type(response.text).__name__}")
try:
    _ = response.text['url']  # спроба key lookup на рядку
except TypeError as e:
    print(f"TypeError: {e}")
print(f"response.json()['url']: {response.json()['url']}")

print()
print("=" * 55)
print("ПИТАННЯ 3: Відповідь B — 401 (HTTP stateless)")
print("=" * 55)
requests.get("https://httpbin.org/basic-auth/user/pass", auth=('user', 'pass'), timeout=5)
response = requests.get("https://httpbin.org/basic-auth/user/pass", timeout=5)
print(f"Status: {response.status_code}")
print("Сервер не пам'ятає першого запиту.")
print("Виправлення: requests.Session() з session.auth = ('user', 'pass')")

print()
print("=" * 55)
print("ПИТАННЯ 4: Відповідь B — ~3 секунди (послідовно)")
print("=" * 55)
import time
start = time.time()
for i in range(3):
    requests.get('https://httpbin.org/delay/1', timeout=10)
print(f"Час: {time.time() - start:.1f}с")
print("Кожен запит блокує Python. Для паралельності потрібен threading або asyncio.")

print()
print("=" * 55)
print("ПИТАННЯ 5: Відповідь B — порожній dict")
print("=" * 55)
response = requests.get("https://httpbin.org/get", data={'q': 'python'}, timeout=5)
print(f"args: {response.json()['args']}  ← порожньо!")
response_correct = requests.get("https://httpbin.org/get", params={'q': 'python'}, timeout=5)
print(f"args з params=: {response_correct.json()['args']}  ← правильно!")

print()
print("=" * 55)
print("ПИТАННЯ 6: Відповідь B — стандартні порти приховані")
print("=" * 55)
import urllib.parse
for url in ['https://google.com', 'http://example.com', 'http://localhost:8000']:
    parsed = urllib.parse.urlparse(url)
    default_port = 443 if parsed.scheme == 'https' else 80
    actual_port = parsed.port or default_port
    print(f"  {url:<35} → порт: {actual_port}")

# 10. Практичні Вправи

---

## Вправа 1: Beginner — The Silent Failure

**Контекст:** Ти пишеш скрипт для завантаження профілів користувачів.
Деякі профілі видалені і повертають 404.

**Завдання:** Запусти код нижче. Спостережи що відбувається.
Потім додай `raise_for_status()` і подивись на різницю.

In [ ]:
import requests

# httpbin.org/status/404 симулює "сторінку не знайдено"
response = requests.get("https://httpbin.org/status/404", timeout=5)

print("Request finished!")
print("Status Code:", response.status_code)

# TODO: Додай response.raise_for_status() після запиту
# і обгорни все у try/except requests.exceptions.HTTPError

# ПИТАННЯ ДЛЯ РОЗДУМІВ:
# Якщо requests не падає при 404 — що станеться якщо ти одразу викличеш
# response.json() і вставиш результат у базу даних?

## Вправа 2: Beginner — The JSON Illusion

In [ ]:
import requests

response = requests.get("https://httpbin.org/get", timeout=5)
data = response.text  # це рядок!

# Спроба отримати URL — TypeError!
# print(data['url'])  # ← розкоментуй і подивись на помилку

# TODO: Виправ код: використай response.json() замість response.text
# і надрукуй значення ключа 'url'

# ПИТАННЯ ДЛЯ РОЗДУМІВ:
# Чому requests надає метод .json(), але не .xml() або .csv()?

## Вправа 3: Intermediate — The Amnesiac Server

In [ ]:
import requests

# Крок 1: Авторизуємось (надсилаємо credentials)
r1 = requests.get(
    "https://httpbin.org/basic-auth/user/pass",
    auth=('user', 'pass'),
    timeout=5
)
print(f"Крок 1 (з auth): {r1.status_code}")

# Крок 2: Намагаємось отримати захищені дані БЕЗ auth
r2 = requests.get("https://httpbin.org/basic-auth/user/pass", timeout=5)
print(f"Крок 2 (без auth): {r2.status_code}")

print()
print("Сервер не пам'ятає нас — HTTP stateless!")

# TODO: Виправ код використавши requests.Session()
# session = requests.Session()
# session.auth = ('user', 'pass')
# і виконай обидва запити через session.get()

# ПИТАННЯ ДЛЯ РОЗДУМІВ:
# Якщо сервер не пам'ятає запити — як браузер тримає тебе залогіненим
# коли ти клікаєш по сторінках Facebook?

## Вправа 4: Intermediate — The Infinite Hang (Blocking)

In [ ]:
import requests
import time

# httpbin.org/delay/1 — сервер навмисно чекає 1 секунду
print("Час виконання 3 послідовних запитів по 1с:")

start = time.time()

for i in range(3):
    print(f"  Запит {i+1}...")
    requests.get('https://httpbin.org/delay/1', timeout=10)

elapsed = time.time() - start
print(f"Загальний час: {elapsed:.1f}с")

# СПОСТЕРЕЖЕННЯ: ~3 секунди, а не ~1 секунда
# Кожен запит БЛОКУЄ Python і чекає відповіді

# ПИТАННЯ ДЛЯ РОЗДУМІВ:
# Ти запускаєш Python web server з 100 користувачами.
# Кожен запит твого сервера викликає зовнішній API (1 секунда).
# Що відбудеться з іншими 99 користувачами поки обробляється перший?

# TODO: Дослідж concurrent.futures.ThreadPoolExecutor
# як засіб для паралельного виконання HTTP запитів

## Вправа 5: Intermediate — HTTP Methods & Bodies

In [ ]:
import requests

# Спроба шукати з data= у GET запиті
payload = {'q': 'python'}

response = requests.get("https://httpbin.org/get", data=payload, timeout=5)
print(f"GET з data=: args = {response.json()['args']}")
print(f"URL відправлено: {response.request.url}")

# TODO: Виправ: змін data= на params=
# Перевір що тепер URL містить ?q=python
# і args = {'q': 'python'}

print()
print("Таблиця: params= vs data= vs json=")
print("params= → URL query string (?q=python) → для GET пошуку")
print("data=   → HTTP body (form-encoded) → для POST форм")
print("json=   → HTTP body (JSON) → для REST API POST/PUT")

# ПИТАННЯ ДЛЯ РОЗДУМІВ:
# Чому GET запити не мають тіла за HTTP специфікацією?
# Яку архітектурну причину мав IETF для цього?

# 11. Real-World Engineering Relevance

## 11.1. Мікросервіси

У сучасних distributed systems незалежні сервіси не мають спільних баз даних.

Вони спілкуються виключно через HTTP:

```text
User Browser
    │ HTTP GET /order/123
    ▼
API Gateway
    │ HTTP GET /orders/123
    ▼
Order Service
    │ HTTP GET /users/456      HTTP GET /items/789
    ▼                          ▼
User Service              Inventory Service
    │                          │
    └──────────┬───────────────┘
               ▼
          Assembled Response
```

Кожна стрілка — це `requests.get()` під капотом.

---

## 11.2. Authentication

HTTP statelessness вирішується через токени у кожному запиті:

```python
# JWT (JSON Web Token) у кожному запиті
session = requests.Session()
session.headers['Authorization'] = f'Bearer {jwt_token}'

# Сервер верифікує токен при КОЖНОМУ запиті
session.get('/api/protected-data')
session.get('/api/user-profile')
```

---

## 11.3. Resilience: Exponential Backoff

Мережі постійно падають. Production системи обгортають HTTP у retry loops:

```python
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter

# Автоматичний retry з exponential backoff
retry_strategy = Retry(
    total=3,
    backoff_factor=1,              # 1с, 2с, 4с між спробами
    status_forcelist=[429, 500, 502, 503, 504],
    respect_retry_after_header=True  # поважаємо Retry-After header
)

session = requests.Session()
session.mount('https://', HTTPAdapter(max_retries=retry_strategy))
```

---

## 11.4. Rate Limiting

Більшість API обмежують кількість запитів:

```python
response = requests.get('https://api.github.com/users', timeout=5)

# Перевіряємо rate limit headers
remaining = response.headers.get('X-RateLimit-Remaining')
reset_at   = response.headers.get('X-RateLimit-Reset')

if response.status_code == 429:
    retry_after = int(response.headers.get('Retry-After', 60))
    print(f"Rate limited! Чекаємо {retry_after}с...")
    time.sleep(retry_after)  # у продакшн — asyncio.sleep або backoff
```

# 12. Типові Помилки

## Топ-10 помилок з HTTP у Python

### 1. Немає timeout
```python
# НЕПРАВИЛЬНО
response = requests.get(url)

# ПРАВИЛЬНО
response = requests.get(url, timeout=(3.05, 10))
```

### 2. Не перевіряти status_code
```python
# НЕПРАВИЛЬНО
data = response.json()  # може впасти або повернути HTML

# ПРАВИЛЬНО
response.raise_for_status()
data = response.json()
```

### 3. response.text замість response.json()
```python
# НЕПРАВИЛЬНО — рядок, не dict
data = response.text
print(data['key'])  # TypeError!

# ПРАВИЛЬНО
data = response.json()
print(data['key'])
```

### 4. data= замість params= у GET
```python
# НЕПРАВИЛЬНО — тіло ігнорується у GET
requests.get(url, data={'q': 'python'})

# ПРАВИЛЬНО — параметри у URL
requests.get(url, params={'q': 'python'})
```

### 5. dict замість json= у POST
```python
# НЕПРАВИЛЬНО — надсилає form-encoded, не JSON
requests.post(url, data={'key': 'value'})

# ПРАВИЛЬНО — надсилає JSON body
requests.post(url, json={'key': 'value'})
```

### 6. Ігнорувати HTTP statelessness
```python
# НЕПРАВИЛЬНО — auth не зберігається
requests.get(url, auth=creds)
requests.get(other_url)  # 401!

# ПРАВИЛЬНО — Session зберігає стан
session = requests.Session()
session.auth = creds
session.get(url)
session.get(other_url)  # OK
```

### 7. Immediate retry loop (Thundering Herd)
```python
# НЕПРАВИЛЬНО — DDoS власного сервера
while True:
    try:
        response = requests.get(url)
        break
    except:
        pass  # одразу знову!

# ПРАВИЛЬНО — exponential backoff
for attempt in range(3):
    try:
        response = requests.get(url, timeout=5)
        break
    except requests.exceptions.RequestException:
        time.sleep(2 ** attempt)  # 1с, 2с, 4с
```

### 8. Не обробляти ConnectionError окремо від HTTPError
```python
# НЕПРАВИЛЬНО — ConnectionError і HTTPError — різні речі!
try:
    response = requests.get(url)
except Exception as e:
    print("Помилка")  # не зрозуміло яка саме

# ПРАВИЛЬНО — диференційована обробка
try:
    response = requests.get(url, timeout=5)
    response.raise_for_status()
except requests.exceptions.Timeout:
    print("Сервер не відповів")     # мережева проблема
except requests.exceptions.ConnectionError:
    print("Мережа недоступна")      # DNS/TCP проблема
except requests.exceptions.HTTPError as e:
    print(f"HTTP помилка: {e}")     # HTTP проблема
```

# 13. Питання для Роздумів (Рефлексія)

---

## Питання 1

Якщо `requests.get()` повністю блокує Python-інтерпретатор до отримання відповіді,
як браузери завантажують 50 зображень для однієї сторінки одночасно без заморозки?

> Підказка: скільки потоків (threads) або goroutines використовує браузер?

---

## Питання 2

Чому мережеві інженери вважають поганою практикою покладатись на HTTP 500
для дебагінгу замість перевірки payload відповіді?

> Підказка: що може містити `response.text` при 500 помилці?

---

## Питання 3

Оскільки HTTP stateless — як shopping website пам'ятає що ти залогінений
при переходах між сторінками через різні `requests.get()` виклики?

> Підказка: що таке cookies і Authorization header?

---

## Питання 4

Ти запускаєш Python web server (FastAPI/Django) з 10 worker threads.
Кожен worker при обробці запиту викликає зовнішній API (5 секунд).
100 користувачів надіслали запити одночасно.

Що відбудеться? Скільки користувачів отримають відповідь швидко?

> Підказка: 10 threads × 5с blocking = ?

---

## Питання 5

Як `requests.Session()` вирішує проблему HTTP statelessness?
Що конкретно Session зберігає між запитами?

> Підказка: cookies, headers, auth...

---

## Питання 6

Ти пишеш 100 HTTP запитів у циклі без затримки. API повертає 429 Too Many Requests.
Ти додаєш `time.sleep(1)` між запитами — але знову отримуєш 429 при наступному запуску.

Що пішло не так? Що таке Thundering Herd problem?

> Підказка: якщо у тебе 100 worker processes і всі роблять `sleep(1)` → `retry` одночасно...

---

# 14. Real-World Case Study: Метеорологічний Data Pipeline

У директорії цього уроку знаходиться реальний production-проєкт — `ogimet-main`.

Це **метеорологічний data pipeline**, який:
1. Завантажує синоптичні телеграми з сайту [ogimet.com](http://www.ogimet.com) для метеостанцій України, Білорусі
2. Декодує їх у структуровані дані (температура, тиск, вологість, вітер)
3. Зберігає у MongoDB кожні 3 години
4. Відкриває REST API (FastAPI) для фільтрації та пошуку

```text
APScheduler (кожні 3 год)
       │
       ▼
TelegramProcessor.get_station_data()
       │ requests.get(ogimet.com, params={block, begin, end})
       ▼
CSV response → pandas DataFrame
       │
       ▼
TelegramMeteoDecoder.decode()
       │
       ▼
MongoDB insert_or_update
       │
       ▼
FastAPI REST endpoints (GET/POST/PUT/DELETE)
```

---

## Навіщо аналізувати цей проєкт?

Тому що в ньому є **саме ті проблеми**, які ми вивчали на цьому уроці:

| Проблема з уроку | Де в проєкті |
| ---------------- | ------------- |
| Пастка 1: Немає timeout | `requests.get(...)`  без `timeout=` |
| Пастка 5: Ghost Identity (User-Agent) | `fake_useragent` — навмисний обхід WAF |
| Пастка 4: Blocking loop | `for station in stations_list:` з синхронними запитами |
| Правильне: params= | `params={'block': wmo, 'begin': ..., 'end': ...}` |
| Правильне: status_code check | `if response.status_code == 200:` |
| CSV, не JSON | `response.text` → `io.StringIO` → `pd.read_csv()` |

Аналіз реального коду — найкращий спосіб закріпити теорію.

## 14.1. Аналіз: `get_station_data()` — що добре і що варто покращити

```python
# З ogimet-main/meteo_telegram/telegram_decode/meteo_ogimet.py

def get_station_data(self, wmo: str, start_date: datetime, end_date: datetime):
    params = {'block': wmo, 'begin': start_date_str, 'end': end_date_str}
    headers = {'User-Agent': get_user_agent()}   # ← рандомний браузерний UA

    response = requests.get(self.request_link, params=params, headers=headers)
    #                                                                    ^
    #                                          НЕМАЄ timeout= ← Пастка 1!

    if response.status_code == 200:              # ← перевірка статусу (добре!)
        try:
            csv_file_like_object = io.StringIO(response.text)  # ← не JSON!
            df = pd.read_csv(csv_file_like_object, names=headers)
            return df
        except Exception as e:
            print("Error:", e)
            return None
    else:
        print(f"Request failed: {response.status_code}")
        return None
```

### Що ПРАВИЛЬНО

**1. `params=` використано коректно**
```python
params = {'block': wmo, 'begin': start_date_str, 'end': end_date_str}
requests.get(url, params=params)
# → http://ogimet.com/cgi-bin/getsynop?block=33088&begin=202409010000&end=202409030000
```
Параметри пошуку правильно йдуть у URL query string.

**2. `User-Agent` підміна — навмисна і обґрунтована**
```python
headers = {'User-Agent': get_user_agent()}  # random браузерний UA
```
ogimet.com блокує стандартний `python-requests/2.x.x`.
Підміна User-Agent — стандартна практика для web scraping.
Це саме Пастка 5 (Ghost Identity) з уроку.

**3. `status_code` перевіряється**
```python
if response.status_code == 200:
```
Краще ніж нічого — але `raise_for_status()` було б надійніше.

**4. CSV через `response.text` — правильно для цього API**
```python
csv_file_like_object = io.StringIO(response.text)
df = pd.read_csv(csv_file_like_object, names=headers)
```
ogimet повертає CSV, не JSON. Тому `response.json()` тут не потрібен.
Це демонструє що завжди треба перевіряти `Content-Type` відповіді.

---

### Що ВАРТО ПОКРАЩИТИ

**1. Відсутній `timeout=` — Пастка 1**
```python
# Поточний код — небезпечний
response = requests.get(self.request_link, params=params, headers=headers)

# Покращений
response = requests.get(
    self.request_link,
    params=params,
    headers=headers,
    timeout=(5.0, 30)   # ogimet може бути повільним
)
```
Якщо ogimet.com не відповідає — APScheduler task завмирає назавжди.

**2. Sequential blocking loop — Пастка 4**
```python
# Поточний код — послідовний
for station in stations_list:      # 35+ станцій!
    df = self.get_station_data(station, ...)   # блокує кожного разу

# Покращений — паралельний
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=10) as executor:
    results = list(executor.map(
        lambda s: self.get_station_data(s, start_date, end_date),
        stations_list
    ))
```
35 станцій × ~2с/запит = **70 секунд** послідовно.
З 10 threads — всього **~7 секунд**.

**3. Широкий `except Exception` приховує помилки**
```python
# Поточний
except Exception as e:
    print("Error processing response as CSV:", e)
    return None

# Покращений — специфічні винятки + retry логіка
except requests.exceptions.Timeout:
    logging.error(f"Timeout for station {wmo}")
except requests.exceptions.ConnectionError:
    logging.error(f"Network error for station {wmo}")
except pd.errors.ParserError:
    logging.error(f"CSV parse error for station {wmo}: {response.text[:100]}")
```

In [ ]:
# === СИМУЛЯЦІЯ: ogimet-стиль запит (без реального ogimet.com) ===
# Використовуємо httpbin.org для демонстрації тих самих патернів

import requests
import io
import time

# --- Симуляція User-Agent підміни (Пастка 5 — Ghost Identity) ---
BROWSER_UA = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

def fetch_station_data_improved(station_id: str) -> dict | None:
    """
    Покращена версія get_station_data():
    - timeout додано
    - raise_for_status() замість if/else
    - специфічні exceptions
    """
    params = {
        'block': station_id,
        'begin': '202409010000',
        'end':   '202409030000'
    }
    headers = {'User-Agent': BROWSER_UA}

    # Симулюємо запит через httpbin (показує params і headers)
    try:
        response = requests.get(
            'https://httpbin.org/get',
            params=params,
            headers=headers,
            timeout=(5.0, 15)   # ← покращення: завжди вказуємо timeout
        )
        response.raise_for_status()   # ← покращення: явна перевірка статусу

        data = response.json()
        return {
            'station_id': station_id,
            'url_sent': data['url'],
            'args_sent': data['args'],
            'user_agent': data['headers']['User-Agent']
        }

    except requests.exceptions.Timeout:
        print(f"[TIMEOUT] Станція {station_id}: сервер не відповів")
    except requests.exceptions.ConnectionError:
        print(f"[CONNECTION] Станція {station_id}: мережа недоступна")
    except requests.exceptions.HTTPError as e:
        print(f"[HTTP ERROR] Станція {station_id}: {e.response.status_code}")

    return None


# --- Демо: один запит ---
print("=== Запит для однієї станції ===")
result = fetch_station_data_improved('33088')
if result:
    print(f"  Station: {result['station_id']}")
    print(f"  URL sent: {result['url_sent']}")
    print(f"  Params:   {result['args_sent']}")
    print(f"  User-Agent: {result['user_agent'][:60]}...")

print()

# --- Демо: sequential loop (поточний код ogimet) ---
stations = ['33088', '33135', '33177']

print("=== Sequential loop (поточний підхід ogimet) ===")
start = time.time()
results_seq = []
for station in stations:
    r = fetch_station_data_improved(station)
    if r:
        results_seq.append(r)
print(f"Час: {time.time() - start:.2f}с для {len(stations)} станцій")
print()

# --- Демо: ThreadPoolExecutor (покращений підхід) ---
from concurrent.futures import ThreadPoolExecutor

print("=== ThreadPoolExecutor (покращений підхід) ===")
start = time.time()
with ThreadPoolExecutor(max_workers=3) as executor:
    results_par = list(executor.map(fetch_station_data_improved, stations))
results_par = [r for r in results_par if r]
print(f"Час: {time.time() - start:.2f}с для {len(stations)} станцій (паралельно)")
print()
print(f"Обидва підходи дали {len(results_seq)} та {len(results_par)} результатів")
print("З 35 станціями різниця стає суттєвою: ~70с vs ~7с")

## 14.2. Аналіз: FastAPI endpoints — REST design у дії

Проєкт `main.py` показує **правильне використання HTTP methods** у REST API:

```python
# GET — отримати ресурс (idempotent, safe)
@app.get("/telegram/{collection}/{id}")
def get_telegram(collection: str, id: str): ...

# POST — створити або запустити процес
@app.post("/download_telegrams")
def download_telegrams(country_code: str): ...

# POST з фільтром — складний пошук через body
@app.post("/filter_telegrams/")
def filter_telegrams(filter: TelegramFilter): ...

# PUT — оновити ресурс
@app.put("/telegram/{collection}/{id}")
def update_telegram(collection: str, id: str, updates: dict): ...

# DELETE — видалити ресурс
@app.delete("/telegram/{collection}/{id}")
def delete_telegram(collection: str, id: str): ...
```

### Архітектурне спостереження: POST для складного пошуку

Цікаво що `/filter_telegrams/` використовує **POST замість GET**:

```python
# Фільтр має багато полів — не поміститься в URL query string
class TelegramFilter(BaseModel):
    country_code: Optional[str]
    station_id: Optional[str]
    date: Optional[str]
    temperature: Optional[float]
    wind_speed: Optional[float]
    # ... 20+ полів
```

Це компроміс: GET ідеальний для простих фільтрів (`?q=python`),
але коли фільтр складний і має 20+ полів — POST з JSON body практичніший.

### Клієнтський код для цього API

Ось як правильно викликати цей FastAPI сервер:

```python
import requests

BASE_URL = 'http://localhost:8000'

# GET — отримати одну телеграму
response = requests.get(
    f'{BASE_URL}/telegram/ua/3450420249218',
    timeout=5
)
response.raise_for_status()
telegram = response.json()

# POST — запустити завантаження
response = requests.post(
    f'{BASE_URL}/download_telegrams',
    params={'country_code': 'ua'},   # FastAPI читає з query params
    timeout=(5, 120)                 # завантаження може зайняти час
)
response.raise_for_status()

# POST — складний пошук
filter_body = {
    'country_code': 'ua',
    'date': '20240901',
    'hour': 18,
    'fields_to_return': ['station_id', 'temperature']
}
response = requests.post(
    f'{BASE_URL}/filter_telegrams/',
    json=filter_body,   # ← json=, не data=
    timeout=10
)
response.raise_for_status()
results = response.json()['results']

# PUT — оновити поле
response = requests.put(
    f'{BASE_URL}/telegram/ua/3450420249218',
    json={'temperature': 26.0},   # ← json=, не data=
    timeout=5
)
response.raise_for_status()

# DELETE — видалити
response = requests.delete(
    f'{BASE_URL}/telegram/ua/3450420249218',
    timeout=5
)
response.raise_for_status()
```

## 14.3. Питання для аналізу проєкту

Подивись на реальний код у `ogimet-main/meteo_telegram/telegram_decode/meteo_ogimet.py`
і дай відповідь на питання:

---

**Питання 1:**
У методі `get_telegrams()` є цикл:
```python
for station in stations_list:
    df = self.get_station_data(station, start_date, end_date)
```
Якщо `stations_list_ua` містить 36 станцій, а кожен запит займає ~2 секунди —
скільки часу займе весь цикл?
Яку технологію треба використати щоб прискорити у 5-10 разів?

> Підказка: `concurrent.futures.ThreadPoolExecutor` або `asyncio` + `aiohttp`

---

**Питання 2:**
Сервіс запускається через APScheduler кожні 3 години:
```python
scheduler.add_job(download_and_process_telegrams, args=['ua'],
                  trigger=CronTrigger(hour='0,3,6,9,12,15,18,21', ...))
```
Якщо один запуск займає 72 секунди (36 станцій × 2с),
а ogimet іноді не відповідає і один station "висить" вічно —
що відбудеться з наступним запуском о 3:00?

> Підказка: APScheduler задача не завершилась, timeout не встановлений...

---

**Питання 3:**
ogimet повертає CSV з headers `['station_id', 'year', 'month', 'day', 'hour', 'minute', 'telegram']`.
Але якщо ogimet поверне 503 Service Unavailable (HTML сторінку) замість CSV —
що зробить `pd.read_csv(io.StringIO(response.text))`?
Як би ти захистив код від цього?

> Підказка: HTML рядок виглядає як `<!DOCTYPE html>...` — що зробить CSV parser?

# 15. Summary: Ключові Висновки

---

## Архітектурне розуміння

```text
requests.get() — це НЕ "звичайна Python функція"

Це оркестрація:
    DNS lookup
    TCP handshake
    TLS negotiation
    HTTP serialization
    OS blocking syscall
    kernel interrupt
    HTTP parsing
    JSON deserialization
```

---

## Mental Model

```text
requests.get() = "I/O boundary"

Python кладе листа у вихідний лоток
ставить будильник
і йде спати

OS Kernel займається фізикою:
packet fragmentation
routing
TCP handshakes

Python прокидається
коли конверт із відповіддю прийшов
```

---

## Production Checklist

- `timeout=(3.05, 10)` — **завжди**
- `raise_for_status()` — після кожного запиту
- `try/except` з роздільними обробниками для `Timeout`, `ConnectionError`, `HTTPError`
- `params=` для GET параметрів
- `json=` для POST тіла (не `data=`)
- `requests.Session()` для кількох запитів підряд
- Перевіряй `response.request.headers` при дебагінгу

---

## Розрізнення помилок

```text
ConnectionError → мережева помилка (DNS, TCP RST, Wi-Fi)
Timeout         → мережа є, але сервер не відповідає
HTTPError (4xx) → твій запит неправильний (403, 404)
HTTPError (5xx) → сервер зламаний
200 OK          → TCP і HTTP успішні, але перевір JSON!
```

---

## Наступні уроки

| Тема | Що вирішує |
| ---- | ---------- |
| `requests.Session()` + Auth | HTTP statelessness |
| `threading` + HTTP | Blocking I/O paралельно |
| `asyncio` + `aiohttp` | Non-blocking HTTP |
| REST API Design | Архітектура endpoints |
| FastAPI | Створення власного HTTP сервера |